In [ ]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def run_agentic_loop(user_input):
    print(f"\n[Agent]: Analyzing input -> '{user_input}'")

    # 1. The Agent's Decision Prompt
    # We ask the LLM to act as a router and decide the category
    system_prompt = """
    You are an intelligent routing agent. Classify the user input into EXACTLY ONE of these categories:
    - TECH (for coding, python, or software questions)
    - JOKE (for humor or funny statements)
    - UNKNOWN (if it fits neither)

    Respond with ONLY the category name. No extra text.
    """

    try:
        # Step A: The LLM evaluates the input
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_input}
            ],
            temperature=0.1
        )

        decision = response.choices[0].message.content.strip().upper()
        print(f"[Agent Decision]: {decision}")

        # Step B: The Python "Loop/Action" based on the AI's decision
        if "TECH" in decision:
            print("[Action Executed]: Routing to Python Senior Engineer Agent...")
            print("Answer: Python is a high-level, interpreted programming language famous for its clean syntax and massive AI ecosystem.")

        elif "JOKE" in decision:
            print("[Action Executed]: Routing to Comedy Agent...")
            print("Answer: Why do programmers prefer dark mode? Because light attracts bugs!")

        else:
            print("[Action Executed]: Default fallback response...")
            print("Answer: I'm not quite sure how to handle that input yet!")

    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    # Test our agent with different inputs!
    run_agentic_loop("How do I write a function in Python?")
    run_agentic_loop("Knock knock, who's there?")

In [ ]:
 import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# 1. Define a real Python function that acts as our "Tool"
def calculate_rectangle_area(length, width):
    """Calculates the area of a rectangle given length and width."""
    print(f"\n[TOOL EXECUTED]: Running python code -> area = {length} * {width}")
    area = length * width
    return json.dumps({"area": area})

def run_agent_with_tools(user_query):
    print(f"\n[User Query]: {user_query}")

    # 2. Define the tool schema so the LLM knows what tools are available
    tools = [
        {
            "type": "function",
            "function": {
                "name": "calculate_rectangle_area",
                "description": "Calculate the area of a rectangle based on length and width.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "length": {"type": "number", "description": "The length of the rectangle."},
                        "width": {"type": "number", "description": "The width of the rectangle."}
                    },
                    "required": ["length", "width"]
                }
            }
        }
    ]

    # 3. First call: Send the query and tool definitions to the LLM
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": user_query}],
        tools=tools,
        tool_choice="auto"
    )

    response_message = response.choices[0].message

    # 4. Check if the LLM decided to call a tool
    if response_message.tool_calls:
        print("[Agent Reason]: The AI decided it needs a tool to solve this!")

        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)

            # Execute the correct local Python function
            if function_name == "calculate_rectangle_area":
                tool_result = calculate_rectangle_area(
                    length=function_args.get("length"),
                    width=function_args.get("width")
                )

                print(f"[Tool Output]: {tool_result}")

                # 5. Second call: Send the tool's result back to the LLM to formulate the final answer
                final_response = client.chat.completions.create(
                    model="openai/gpt-oss-20b",
                    messages=[
                        {"role": "user", "content": user_query},
                        response_message, # Include the assistant's tool call request
                        {
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "name": function_name,
                            "content": tool_result
                        }
                    ]
                )

                print("\n--- Final Agent Response ---")
                print(final_response.choices[0].message.content)
    else:
        print("The AI answered directly without using a tool:")
        print(response_message.content)

if __name__ == "__main__":
    # Test our tool-using agent!
    run_agent_with_tools("What is the area of a rectangle that has a length of 45 meters and a width of 12 meters?")